In [ ]:
!pip install --upgrade pip --quiet

In [ ]:
!pip install nest_asyncio langchain-community langchain unstructured beautifulsoup4 newspaper3k tiktoken html2text sentence-transformers faiss-cpu llama-cpp-python fastapi uvicorn nbclient nbformat --quiet

In [ ]:
#!unzip hypershift-docs.zip -d hypershift-docs

In [ ]:
# import nest_asyncio
# nest_asyncio.apply()

In [ ]:
from langchain.document_loaders import DirectoryLoader, UnstructuredHTMLLoader

loader = DirectoryLoader(
    path="hypershift-docs",
    glob="**/*.html",
    loader_cls=UnstructuredHTMLLoader
)

docs = loader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)
texts = splitter.split_documents(docs)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

db = FAISS.from_documents(texts, embedding_model)

db.save_local("faiss_store")

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_store", embedding_model, allow_dangerous_deserialization=True)

In [ ]:
from langchain.llms import LlamaCpp

llm = LlamaCpp(
    model_path="./mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    temparature=0.7,
    max_tokens=256,
    verbose=False
)


In [ ]:
from langchain.chains import RetrievalQA

retriever = db.as_retriever(search_kwargs={"k":3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
# query = "How do I install Hypershift?"

# result = qa_chain(query)

# print(result['result'])

In [ ]:
def answer_query(question):
    result = qa_chain.invoke(question)
    return result['result']

# query = "How do I install Hypershift on Azure AKS?"
# answer_query(query)

In [ ]:
# query = "Heyy team, we find the image registry.ci.openshift.org/hypershift/hypershift-operator:latest is not an multi-arch image, but amd64 only. Do we know why the aarch64 version was removed, and any expectation to make it multiarch?"
# print(answer_query(query))

In [ ]:
#print(answer_query(question))